# 03_step1_Ecearth3historical_BA_F1 - Calibration of F1 (global, 1 deg)   [PLAN A]

**Purpose**: learn the function **F1** that turns EC-Earth3 historical into an "ERA5-like" climate at 1 deg.

F1 = **WDF** (dry days, precipitation only) then **QDM** (per month and per pixel).

| variable | treatment |
|---|---|
| `pr` | fixed WDF, then multiplicative QDM |
| `tasmax` | additive QDM |
| `tasmin` | through the diurnal range `DTR = tasmax - tasmin` (multiplicative QDM); `tasmin = adjusted tasmax - adjusted DTR` (guarantees tasmin < tasmax) |

**Fixed WDF rule**: estimated **once** here (per month and per pixel: model threshold, activation probability
of a dry day, seed) and applied **unchanged** to all runs. The dry-day frequency of ERA5 is therefore never
re-imposed on a future run: the hosing signal on dry days is not erased.

Inputs: outputs of notebook 02 (ERA5 and EC-Earth3 historical at 1 deg). Outputs: parameters in `F1/`.

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

CAL_PERIOD = ("1981", "2014")           # calibration period (EC-Earth3 historical ends in 2014)
CHUNKS_SPACE = {"lat": 30, "lon": 60}   # dask spatial blocks (time = a single block, required by QDM)
REGRID_DIR = bu.out_dir(cfg, "plan_A", "regrid_1deg")
F1_DIR = bu.out_dir(cfg, "plan_A", "F1")

## 1. Loading and checks of the calibration pair

In [ ]:
def load_regridded(name):
    d = {}
    for v in VARS:
        d[v] = bu.subset_time(bu.read_processed(REGRID_DIR / f"{name}_{v}_1deg.nc", v), *CAL_PERIOD)
    return d

ref = load_regridded("era5")             # target
hist = load_regridded("gcm_historical")  # to correct
for v in VARS:
    assert bu.same_grid(ref[v], hist[v]), f"{v}: different grids (rerun notebook 02)"
    print(f"{v:7s} ERA5 {ref[v].sizes['time']} days | GCM {hist[v].sizes['time']} days | grid {ref[v].sizes['lat']}x{ref[v].sizes['lon']}")
print("NaN pixels in ERA5 (pr):", int(ref['pr'].isel(time=0).isnull().sum().compute()))

## 2. Calibration (WDF then QDM)

In [ ]:
bu.fit_F(ref, hist, F1_DIR, chunks_space=CHUNKS_SPACE)
print(sorted(p.name for p in F1_DIR.iterdir()))

## 3. A look at the WDF rule

`p0_ref` / `p0_hist`: dry-day fraction of ERA5 / of the GCM; `t`: threshold applied to the GCM (above 0.1 mm if the GCM is too wet);
`q`: probability of activating a GCM dry day (if the GCM is too dry).

In [ ]:
par = xr.open_dataset(F1_DIR / "wdf_params.nc").load()
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for r, m in enumerate([1, 7]):
    for c, (k, kw) in enumerate([("p0_ref", dict(vmin=0, vmax=1)), ("p0_hist", dict(vmin=0, vmax=1)),
                                 ("t", dict(vmin=0.1, vmax=1.0)), ("q", dict(vmin=0, vmax=0.5))]):
        a = par[k].sel(month=m)
        im = axes[r, c].pcolormesh(a.lon, a.lat, a, shading="auto", **kw); plt.colorbar(im, ax=axes[r, c], shrink=0.7)
        axes[r, c].set_title(f"{k} - month {m}")
plt.tight_layout(); plt.savefig(F1_DIR / "wdf_rule.png", dpi=110); plt.show()

## 4. In-sample check (global, first 10 years)

F1 is applied to EC-Earth3 historical itself: the quantiles should fall back onto those of ERA5.
This is **not** a validation (same period as the training): it only checks that the code does what it should.
The real validation (separate periods) is in notebook 09.

In [ ]:
sample_idx = slice(0, 365 * 10)   # 10 years, whole globe
Rc = {v: x.isel(time=sample_idx).compute() for v, x in ref.items()}
Hc = {v: x.isel(time=sample_idx).compute() for v, x in hist.items()}
Ac = xr.Dataset(bu.apply_F(Hc, F1_DIR, chunks_space=CHUNKS_SPACE)).compute()   # single computation
rows = []
for v in VARS:
    for lab, x in (("ERA5 (target)", Rc[v]), ("raw GCM", Hc[v]), ("GCM adjusted F1", Ac[v])):
        s = bu.summarize(x)
        rows.append({"var": v, "series": lab, **{k: round(val, 3) for k, val in s.items() if k != "n"}})
pd.DataFrame(rows)